# 종목의 신원을 잇다 — 코드 하나로는 아무 데도 못 붙는다

> `notebooks/01-데이터수집/05.종목의신원을잇다.ipynb` · 2026-09-03 · 이동원
> 마이그레이션 **v10** · 설계 [`docs/데이터파트/version3.2/공공데이터포털_수집_설계.md`]

---

## 이 노트북이 답하는 것

> **"우리 시세 922만 행은 종목코드만 아는데, DART 재무나 해외 자료에 어떻게 붙이나?"**

`daily_price` 는 종목을 **KRX 종목코드**(`000020`)로만 압니다. 그런데 —

| 붙이고 싶은 것 | 그쪽이 쓰는 열쇠 | 우리가 가진 것 |
|---|---|---|
| DART 재무 662,933행 | 고유번호(`00119195`) | 종목코드 |
| 해외 자료·지수 편입 | ISIN(`KR7000020008`) | 종목코드 |
| 법인 등기·감사 정보 | 법인등록번호(`1101110043870`) | 종목코드 |

**열쇠가 다 다릅니다.** 그래서 공공데이터포털 금융위 API 에서 **다리**를 받아 왔습니다.

시세는 안 받습니다 — 포털 주식시세는 20칸뿐이고 KRX 가 정본입니다. 재무도 안 받습니다 —
DART 가 계정과목 단위로 훨씬 자세합니다. **겹치는 것을 더하면 관리 비용만 늘고, 두
출처가 어긋날 때 어느 쪽이 옳은지 정하는 일이 새로 생깁니다.**

---

## 이 노트북에서 가장 중요한 한 가지

설계를 다 써 놓고 구현하러 API 를 실제로 불러 보니, **설계가 몰랐던 것**이 나왔습니다.

> 🔴 **포털의 `basDt` 목록은 그 시점의 상장 목록이 아닙니다.**

`basDt=20200102` 로 받은 2,334종 안에 **그날 이후에야 상장된 종목이 33종** 있었습니다.
아래에서 직접 세어 보겠습니다.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1] if Path.cwd().name.startswith("0") else Path.cwd()))

from ingest.store.krx_store import connect

with connect() as conn:
    표본날 = "20200102"
    포털 = {r[0]: r[1] for r in conn.execute(
        "SELECT code, item_nm FROM stock_identity WHERE bas_dd = ?", (표본날,))}
    늦은것 = []
    for 코드, 이름 in 포털.items():
        첫날 = conn.execute(
            "SELECT MIN(bas_dd) FROM daily_price WHERE code = ?", (코드,)).fetchone()[0]
        if 첫날 and 첫날 > 표본날:
            늦은것.append((코드, 이름, 첫날))

print(f"포털이 {표본날} 목록이라며 준 종목 : {len(포털):,}종")
print(f"그중 그날 이후에야 상장된 종목    : {len(늦은것):,}종\n")
for 코드, 이름, 첫날 in sorted(늦은것, key=lambda x: x[2])[-5:]:
    해차 = int(첫날[:4]) - int(표본날[:4])
    print(f"  {코드}  {이름:16s} 첫 시세 {첫날}  ({해차}년 뒤)")

포털이 20200102 목록이라며 준 종목 : 2,334종
그중 그날 이후에야 상장된 종목    : 33종

  221800  유투바이오            첫 시세 20231102  (3년 뒤)
  323350  다원넥스뷰            첫 시세 20240611  (4년 뒤)
  107640  한중엔시에스           첫 시세 20240624  (4년 뒤)
  044990  에이치엔에스하이텍        첫 시세 20241025  (4년 뒤)
  176750  듀켐바이오            첫 시세 20241220  (4년 뒤)


**듀켐바이오는 첫 시세가 4년 뒤인 2024-12-20 입니다.**

포털은 최신에 가까운 목록에 기준일 딱지만 붙여 주는 것으로 보입니다. 이걸 모르고
`stock_identity` 로 유니버스를 만들면 **아직 상장도 안 된 종목이 섞입니다** —
미래참조이고, **에러가 안 나고 성능만 좋아집니다.**

    유니버스를 만들 땐 반드시 `daily_price` 와 **교집합**을 냅니다.

이 표의 쓸모는 *"그날 무엇이 상장돼 있었나"* 가 아니라 **다리**입니다.
다리로 쓰는 데는 이 성질이 해가 되지 않습니다.

---

## 무엇이 담겼나

In [2]:
from ingest.store import identity_store as ids

상태 = ids.status()
신원, 개요 = 상태["stock_identity"], 상태["corp_profile"]
print(f"stock_identity : {신원['rows']:>8,}행 · {신원['days']:,}일 · {신원['codes']:,}종목")
print(f"                 {신원['first']} ~ {신원['last']}")
print(f"corp_profile   : {개요['rows']:>8,}행 · 법인 {개요['crno']:,}곳")
print(f"                 유효시작 {개요['first']} ~ {개요['last']}")

stock_identity :    7,000행 · 3일 · 2,334종목
                 20200102 ~ 20200106
corp_profile   :       82행 · 법인 5곳
                 유효시작 20200509 ~ 20260701


In [3]:
import pandas as pd

with connect() as conn:
    df = pd.read_sql_query(
        "SELECT bas_dd, code, item_nm, corp_nm, market, isin_cd, crno, known_at "
        "FROM stock_identity WHERE bas_dd='20200102' ORDER BY code LIMIT 5", conn)
df

,bas_dd,code,item_nm,corp_nm,market,isin_cd,crno,known_at
0,20200102,000020,동화약품,동화약품(주),KOSPI,KR7000020008,1101110043870,20200103
1,20200102,000040,KR모터스,KR모터스(주),KOSPI,KR7000040006,1601110007032,20200103
2,20200102,000050,경방,(주)경방,KOSPI,KR7000050005,1101110013287,20200103
3,20200102,000060,메리츠화재,메리츠화재해상보험(주),KOSPI,KR7000060004,1101110013328,20200103
4,20200102,000070,삼양홀딩스,(주)삼양홀딩스,KOSPI,KR7000070003,1101110026181,20200103


**`code` 에 `A` 가 없습니다.** 원문은 `A000020` 으로 옵니다.

> 🔴 이 접두사를 안 떼면 `daily_price.code` 와 조인이 **0행**이 됩니다. 그런데
> **조인은 0행이어도 에러가 나지 않습니다** — 그냥 아무것도 안 나옵니다.
> 그래서 스키마 CHECK 로도 막았습니다.

`known_at` 이 `bas_dd` 보다 하루 뒤인 것도 보입니다. 포털은 *"기준일자로부터 영업일
하루 뒤 오후 1시 이후 갱신"* 이라, 기준일 당일에는 이 자료를 볼 수 없었습니다.

---

## 법인 개요 — 여기서는 시점이 **관측값**입니다

설계는 `corp_profile` 의 기본키를 `(crno, bas_dd)` 로 잡았습니다. 그런데 응답에
`basDt` 칸이 **아예 없었습니다.** 대신 `fstOpegDt`~`lastOpegDt` 가 그 스냅샷의
**유효구간**이고, 한 법인의 여러 행이 겹치지 않게 이어집니다.

In [4]:
with connect() as conn:
    한법인 = pd.read_sql_query(
        "SELECT fst_opeg_dt, last_opeg_dt, empe_cnt, pn1_avg_slry_amt, "
        "audt_rpt_opnn, actn_audpn FROM corp_profile "
        "WHERE crno = (SELECT crno FROM corp_profile GROUP BY crno "
        "              ORDER BY COUNT(*) DESC LIMIT 1) "
        "ORDER BY fst_opeg_dt", conn)
print(f"한 법인의 이력 {len(한법인)}행 — 구간이 겹치지 않는다")
한법인

한 법인의 이력 24행 — 구간이 겹치지 않는다


,fst_opeg_dt,last_opeg_dt,empe_cnt,pn1_avg_slry_amt,audt_rpt_opnn,actn_audpn
0,20200509,20201102,5575,68000000,적정의견,삼일회계법인
1,20201103,20210318,5575,68000000,적정의견,삼일회계법인
2,20210319,20210329,4972,70000000,적정의견,삼일회계법인
3,20210330,20220320,4972,70000000,적정의견,삼일회계법인
4,20220321,20220703,4751,76000000,적정의견,삼일회계법인
5,20220704,20220904,4751,76000000,적정의견,삼일회계법인
6,20220905,20220928,4751,76000000,적정의견,삼일회계법인
7,20220929,20230321,4751,76000000,적정의견,삼일회계법인
8,20230322,20230329,4927,85000000,예외사항없음,삼일회계법인
9,20230330,20230502,4927,85000000,예외사항없음,삼일회계법인


이게 우리에게 **이득**입니다.

거시(ECOS)는 발표일을 주지 않아 `known_at` 을 *"기준일의 다음 영업일"* 로 계산할
수밖에 없었습니다. 계산값이라 **규칙을 바꾸면 다시 받아야 하는 짐**이 남았습니다.

여기서는 출처가 직접 *"이 값은 이 날부터 유효했다"* 를 말해 줍니다. 그 짐이 없습니다.

| 표 | `known_at` | 성질 |
|---|---|---|
| `stock_identity` | `bas_dd` 의 다음 거래일 | 🔴 계산값 |
| `corp_profile` | `fstOpegDt` 그대로 | ✅ **관측값** |

---

## 상장폐지일 — 시세로는 알 수 없던 것

지금 우리는 *"어느 날부터 시세가 안 나온다"* 로 폐지를 추정합니다. 그런데 그건
**장기 거래정지와 구별되지 않습니다.** 포털은 공식 폐지일을 줍니다.

In [5]:
with connect() as conn:
    폐지 = pd.read_sql_query(
        "SELECT corp_nm, xchg_lstg_dt, xchg_lstg_abol_dt FROM corp_profile "
        "WHERE xchg_lstg_abol_dt IS NOT NULL "
        "GROUP BY crno ORDER BY xchg_lstg_abol_dt", conn)
print(f"폐지일이 있는 법인 {len(폐지)}곳")
폐지

폐지일이 있는 법인 1곳


,corp_nm,xchg_lstg_dt,xchg_lstg_abol_dt
0,한국제지(주),None,20200713


원문은 `23/02/21` 처럼 **두 자리 연도**로 옵니다. 한 응답 안에 날짜 형식이
**세 가지**입니다.

```
enpEstbDt          18970925      YYYYMMDD
enpXchgLstgDt      76/03/24      YY/MM/DD     ← 두 자리 연도
fssCorpChgDtm      2025/08/07    YYYY/MM/DD
```

`56~99 → 1900대` · `00~55 → 2000대` 로 풀었습니다. 근거는 도메인입니다 —
**KRX 가 1956년 3월 개장**이라 그보다 이른 상장일이 없습니다.

> "올해를 기준으로" 같은 규칙은 쓰지 않았습니다. 해가 바뀌면 **같은 원문이 다른
> 값으로** 읽혀서, 작년에 받은 행과 올해 받은 행이 조용히 어긋납니다.

In [6]:
from ingest.clients.data_go_kr import normalize_date

for 원문 in ["18970925", "2025/08/07", "76/03/24", "23/02/21", "55/12/31", "56/01/01", "", "abc"]:
    print(f"  {원문!r:14s} → {normalize_date(원문)!r}")

  '18970925'     → '18970925'
  '2025/08/07'   → '20250807'
  '76/03/24'     → '19760324'
  '23/02/21'     → '20230221'
  '55/12/31'     → '20551231'
  '56/01/01'     → '19560101'
  ''             → None
  'abc'          → None


---

## 무엇이 안 담기나 — 검증기가 갈라 준 것

처음 검증기를 돌렸을 때 *"영문 낀 코드가 시세엔 84종인데 신원엔 0종"* 이라며
빨간불이 떴습니다. 우리가 숫자로 걸렀나 싶었는데, 갈라 보니 아니었습니다.

In [7]:
with connect() as conn:
    구간 = conn.execute("SELECT MIN(bas_dd), MAX(bas_dd) FROM stock_identity").fetchone()
    시세 = {r[0]: r[1] for r in conn.execute(
        "SELECT code, MAX(name) FROM daily_price WHERE bas_dd BETWEEN ? AND ? GROUP BY code", 구간)}
    신원 = {r[0] for r in conn.execute("SELECT DISTINCT code FROM stock_identity")}

못이은 = {c: n for c, n in 시세.items() if c not in 신원}
우선주 = {c: n for c, n in 못이은.items()
        if n and (n.endswith("우") or "우B" in n or "우(전환)" in n or n.endswith("우C"))}
외국 = {c: n for c, n in 못이은.items() if c.startswith(("900", "950")) and c not in 우선주}
그밖 = {c: n for c, n in 못이은.items() if c not in 우선주 and c not in 외국}

print(f"시세 {len(시세):,}종 · 신원 {len(신원):,}종 · 못 이은 {len(못이은):,}종")
print(f"  우선주   {len(우선주):>4,}  ← 포털이 안 준다")
print(f"  외국기업 {len(외국):>4,}  ← 포털이 안 준다")
print(f"  그 밖    {len(그밖):>4,}  ← 여기가 커지면 우리 잘못")

시세 2,324종 · 신원 2,334종 · 못 이은 141종
  우선주    120  ← 포털이 안 준다
  외국기업   21  ← 포털이 안 준다
  그 밖       0  ← 여기가 커지면 우리 잘못


**포털의 목록이 우선주와 외국기업을 아예 안 줍니다.** 신형우선주 84종은 그
우선주 안에 들어 있었습니다.

> 우리가 거른 것과 출처가 안 준 것을 섞으면 **없는 버그를 쫓게 됩니다.**
> 그래서 검사를 갈랐습니다 — "우선주·외국기업 말고 못 이은 것" 이 5% 를 넘을
> 때만 실패로 봅니다. 지금은 **0%** 입니다.

검증기가 거짓 경보를 내면 사람이 곧 무시합니다. 그래서 **알려진 사실은 사실로
보고하고, 판정은 우리가 책임질 수 있는 것에만** 겁니다.

---

## 얼마나 받아야 하나

설계는 *"전 구간을 하루씩 훑으면 12,306콜이라 하루 한도 10,000 을 넘는다"* 고 보고
월 단위 샘플링을 검토했습니다. **전제가 틀렸습니다.**

In [8]:
from common.trading_calendar import load_session_days
from ingest.clients import data_go_kr

달력 = sorted(load_session_days())
전구간 = [d for d in 달력 if d >= data_go_kr.EARLIEST_BAS_DD]
설계가정 = len(달력)

print(f"설계가 가정한 거래일 : {설계가정:,}일 → {설계가정 * 3:,}콜  🔴 한도 초과")
print(f"실제 있는 거래일     : {len(전구간):,}일 → "
      f"{data_go_kr.estimate_calls(전구간):,}콜  ✅ 한도 안")
print(f"\n포털이 주는 가장 이른 날 : {data_go_kr.EARLIEST_BAS_DD}")
print("2019 이전은 전부 totalCount=0 이다 — 우리 시세 2010~2019 의 10년은 못 채운다")

설계가 가정한 거래일 : 4,102일 → 12,306콜  🔴 한도 초과
실제 있는 거래일     : 1,636일 → 4,908콜  ✅ 한도 안

포털이 주는 가장 이른 날 : 20200102
2019 이전은 전부 totalCount=0 이다 — 우리 시세 2010~2019 의 10년은 못 채운다


과거가 2020년부터라 거래일이 **1,636일**뿐입니다. 샘플링할 이유가 없어졌고,
매일 받으면 신규상장·폐지 날짜를 거래일 단위로 정확히 잡습니다.

`EARLIEST_BAS_DD` 를 상수로 둬서 그보다 이르면 **호출조차 하지 않습니다.**
모르고 2010년부터 훑으면 2,500콜을 0건에 쓰고 하루 한도의 4분의 1이 사라집니다.

---

## 곁다리로 고친 것 — `next_session()` 이 422배 빨라졌다

`known_at` 은 "기준일의 다음 거래일" 이라 **행마다** 달력을 묻습니다. 그런데 그
함수가 4,102개를 매번 **선형 스캔**하고 있었습니다.

In [9]:
import time

from common import trading_calendar as tc

days = tc.load_session_days()
정렬 = sorted(days)

def next_session_옛(bas_dd):
    return min(d for d in days if d > bas_dd)

표본 = 정렬[::4][:1000]
t0 = time.perf_counter()
for d in 표본:
    tc.next_session(d)
새 = time.perf_counter() - t0

t0 = time.perf_counter()
for d in 표본:
    next_session_옛(d)
옛 = time.perf_counter() - t0

print(f"{len(표본):,}번 호출")
print(f"  옛(선형 스캔) : {옛*1000:8.1f} ms")
print(f"  새(이분탐색)  : {새*1000:8.1f} ms   → {옛/새:.0f}배")
어긋남 = sum(1 for d in 정렬[:-1] if tc.next_session(d) != next_session_옛(d))
print(f"\n답이 어긋난 날짜 : {어긋남:,}  (0이어야 한다)")

1,000번 호출
  옛(선형 스캔) :    328.2 ms
  새(이분탐색)  :      1.2 ms   → 283배



답이 어긋난 날짜 : 0  (0이어야 한다)


**빠른 것보다 답이 같은 게 먼저입니다.** 그래서 옛 구현을 지우지 않고 되살려
전 날짜를 맞대어 봤습니다. 기대값이 새 코드에서 나오면 항등식이라 아무것도 못 잡습니다.

한 가지 더 조심할 것이 있었습니다 — 정렬본은 집합에서 파생된 **사본**이라,
집합만 갈아 끼우면 **조용히 옛 달력으로 답합니다.** 테스트가 실제로 그렇게 합니다.
그래서 갱신 책임을 갈아 끼우는 쪽에 맡기지 않고 **읽는 쪽 한 곳**에서 지키게 했고,
일부러 망가뜨려 회귀 테스트가 잡는 것을 확인했습니다.

---

## 정리 — 설계가 실측에 부딪힌 자리

v3.1 설계를 쓸 때도 실측을 했습니다. 그런데 **엔드포인트가 열려 있는지**까지만 재고,
**응답이 무엇을 담고 있는지**는 한 건도 안 봤습니다.

| 설계 | 실제 | 무엇이 갈렸나 |
|---|---|---|
| 전 구간 4,102 거래일 | **1,636일** (2020~) | 12,306콜 → 4,908콜. 샘플링 불필요 |
| PK `(crno, bas_dd)` | **`(crno, fst_opeg_dt)`** | `basDt` 가 없다. `known_at` 이 관측값이 됐다 |
| 칸 이름 `estb_dt`… | **`enpEstbDt`…** | `enp` 접두사. 틀리면 조용히 NULL |
| (물어볼 생각도 못 함) | **`basDt` 목록이 시점 목록이 아니다** | 유니버스에 미래 종목이 섞인다 |

**"열려 있나" 와 "무엇이 들어 있나" 는 다른 질문이고, 설계는 뒤엣것 위에 서야 합니다.**
앞엣것만 재고 설계하면 문서가 그럴듯한 채로 틀립니다.

---

## 다시 해 보려면

```bash
python scripts/fetch_data_go_kr.py --plan               # 몇 콜 쓸지 먼저 센다
python scripts/fetch_data_go_kr.py --listed --limit 3   # 3일치
python scripts/fetch_data_go_kr.py --profile --limit 5  # 법인 5곳
python scripts/verify_identity.py                       # 값 검증
```